# 第 02 期：Logistic 回归与滑坡易发性

本期用滑坡易发性评价作为案例，讲解 Logistic 回归的基本原理、实现方法和地学解释。

## 1. 导入必要的库

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, 
    classification_report,
    roc_curve, 
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

EPISODE_DIR = Path.cwd()
if not (EPISODE_DIR / 'figures').exists() and Path('episodes/02-logistic-regression/figures').exists():
    EPISODE_DIR = Path('episodes/02-logistic-regression')

FIGURE_DIR = EPISODE_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)


## 2. Sigmoid 函数可视化

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 1000)
p = sigmoid(z)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(z, p, 'b-', linewidth=2, label='Sigmoid 函数')
ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='p = 0.5')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axhline(y=1, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('z (线性组合)', fontsize=12)
ax.set_ylabel('p (概率)', fontsize=12)
ax.set_title('Sigmoid 函数曲线', fontsize=14)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'sigmoid_function.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## 3. 构造教学数据

In [ ]:
np.random.seed(42)
n_samples = 800

slope = np.random.uniform(5, 45, n_samples)
elevation = np.random.uniform(500, 2500, n_samples)
fault_dist = np.random.uniform(50, 3000, n_samples)
road_dist = np.random.uniform(20, 1500, n_samples)
ndvi = np.random.uniform(0.1, 0.9, n_samples)

# 教学构造关系：坡度越大风险越高；距离断层/道路越远风险越低；
# NDVI 越高风险越低；高程在该示例中设为弱负向关系。
log_odds = (
    -1.5 +
    0.10 * slope +
    -0.0002 * elevation +
    -0.0012 * fault_dist +
    -0.0015 * road_dist +
    -2.2 * ndvi
)

prob = sigmoid(log_odds)
landslide = (np.random.random(n_samples) < prob).astype(int)

data = pd.DataFrame({
    'slope': slope,
    'elevation': elevation,
    'fault_dist': fault_dist,
    'road_dist': road_dist,
    'ndvi': ndvi,
    'landslide': landslide
})

print("数据概览：")
print(data.head(10))
print(f"\n数据形状: {data.shape}")
print(f"\n滑坡点数量: {landslide.sum()} ({landslide.mean()*100:.1f}%)")
print(f"非滑坡点数量: {n_samples - landslide.sum()} ({(1-landslide.mean())*100:.1f}%)")


## 4. 数据探索与可视化

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features = ['slope', 'elevation', 'fault_dist', 'road_dist', 'ndvi']
feature_names = ['坡度 (°)', '高程 (m)', '断层距离 (m)', '道路距离 (m)', 'NDVI']

for idx, (feat, name) in enumerate(zip(features, feature_names)):
    ax = axes[idx // 3, idx % 3]
    ax.hist(data[data['landslide']==0][feat], bins=30, alpha=0.6, label='非滑坡点', color='blue')
    ax.hist(data[data['landslide']==1][feat], bins=30, alpha=0.6, label='滑坡点', color='red')
    ax.set_xlabel(name, fontsize=11)
    ax.set_ylabel('频数', fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[1, 2].axis('off')
fig.suptitle('滑坡点与非滑坡点在各环境因子上的分布', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'landslide_factors_scatter.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## 5. 数据预处理与训练集划分

In [ ]:
X = data[features]
y = data['landslide']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"训练集大小: {X_train.shape[0]}")
print(f"测试集大小: {X_test.shape[0]}")
print(f"\n训练集滑坡比例: {y_train.mean()*100:.1f}%")
print(f"测试集滑坡比例: {y_test.mean()*100:.1f}%")


## 6. 训练 Logistic 回归模型

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("="*50)
print("模型参数")
print("="*50)
print(f"截距 (Intercept): {model.intercept_[0]:.4f}")
print("\n各特征系数：")
for feat, coef in zip(features, model.coef_[0]):
    print(f"  {feat}: {coef:.4f}")

print("\n" + "="*50)
print("几率比 (Odds Ratio)")
print("="*50)
for feat, coef in zip(features, model.coef_[0]):
    or_value = np.exp(coef)
    print(f"  {feat}: {or_value:.4f}")


## 7. 模型性能评估

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("="*50)
print("分类性能指标")
print("="*50)
print(f"准确率 (Accuracy): {accuracy:.4f}")
print(f"精确率 (Precision): {precision:.4f}")
print(f"召回率 (Recall): {recall:.4f}")
print(f"F1 分数: {f1:.4f}")
print(f"AUC: {auc:.4f}")

print("\n" + "="*50)
print("分类报告")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['非滑坡', '滑坡']))


## 8. 混淆矩阵

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['预测非滑坡', '预测滑坡'])
ax.set_yticklabels(['实际非滑坡', '实际滑坡'])

for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=14)

ax.set_xlabel('预测标签', fontsize=12)
ax.set_ylabel('真实标签', fontsize=12)
ax.set_title('混淆矩阵', fontsize=14)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

tn, fp, fn, tp = cm.ravel()
textstr = f'TN={tn}  FP={fp}\nFN={fn}  TP={tp}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
ax.text(1.18, 0.5, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='center', bbox=props)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## 9. ROC 曲线

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC 曲线 (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='随机猜测 (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.3)
ax.set_xlabel('假正例率 (FPR)', fontsize=12)
ax.set_ylabel('真正例率 (TPR)', fontsize=12)
ax.set_title('ROC 曲线', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## 10. 模拟滑坡易发性图

In [ ]:
n_grid = 100
slope_grid = np.linspace(data['slope'].min(), data['slope'].max(), n_grid)
fault_dist_grid = np.linspace(data['fault_dist'].min(), data['fault_dist'].max(), n_grid)

SLOPE, FAULT_DIST = np.meshgrid(slope_grid, fault_dist_grid)

mean_elevation = data['elevation'].mean()
mean_road_dist = data['road_dist'].mean()
mean_ndvi = data['ndvi'].mean()

grid_data = pd.DataFrame({
    'slope': SLOPE.flatten(),
    'elevation': mean_elevation,
    'fault_dist': FAULT_DIST.flatten(),
    'road_dist': mean_road_dist,
    'ndvi': mean_ndvi
})

grid_scaled = scaler.transform(grid_data)
grid_prob = model.predict_proba(grid_scaled)[:, 1]
PROB = grid_prob.reshape(SLOPE.shape)

fig, ax = plt.subplots(figsize=(10, 8))
contour = ax.contourf(SLOPE, FAULT_DIST, PROB, levels=20, cmap='RdYlGn_r')
cbar = plt.colorbar(contour, ax=ax)
cbar.set_label('滑坡概率', fontsize=12)
ax.set_xlabel('坡度 (°)', fontsize=12)
ax.set_ylabel('断层距离 (m)', fontsize=12)
ax.set_title(f'滑坡易发性概率图\n(高程={mean_elevation:.0f}m, 道路距离={mean_road_dist:.0f}m, NDVI={mean_ndvi:.2f})', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'susceptibility_map.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## 11. 系数解释与总结

In [ ]:
print("="*60)
print("模型系数解释")
print("="*60)

coef_df = pd.DataFrame({
    '特征': features,
    '系数': model.coef_[0],
    '几率比': np.exp(model.coef_[0]),
    '影响方向': ['增加风险' if c > 0 else '降低风险' for c in model.coef_[0]]
})

print(coef_df.to_string(index=False))

print("\n" + "="*60)
print("地学解释")
print("="*60)
print("""
1. 坡度 (slope): 系数为正，坡度越大滑坡风险越高
   - 符合物理直觉：陡坡更容易发生滑坡

2. 高程 (elevation): 系数取决于区域特征
   - 不同区域可能有不同表现

3. 断层距离 (fault_dist): 系数为负，距断层越近风险越高
   - 断层附近岩体破碎，稳定性差

4. 道路距离 (road_dist): 系数为负，距道路越近风险越高
   - 道路开挖改变坡体应力状态

5. NDVI: 系数为负，植被覆盖越高风险越低
   - 植被根系固土，减少侵蚀
""")

print("\n" + "="*60)
print("本期要点总结")
print("="*60)
print("""
1. Logistic 回归适合二分类问题，输出概率估计
2. Sigmoid 函数将线性组合映射到 [0, 1] 区间
3. 系数可解释为因子对事件发生的影响方向和强度
4. 评估分类模型需关注混淆矩阵、ROC 曲线和 AUC
5. 阈值选择需要权衡漏报和误报的代价
""")
